In [2]:
import os
import sys
from pathlib import Path

try:
    # En Google Colab: monta Drive y usa la copia del repositorio que está ahí.
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/TP-PSI-G8-main")
    os.chdir(ROOT)
except ImportError:
    # Local (VS Code / nbconvert): la carpeta del repositorio.
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

assert (ROOT / "src" / "pipeline.py").exists(), "Revisá la ruta del repositorio actualizado."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Proyecto:", ROOT)

Proyecto: c:\Users\leahw\PSIB\TP-PSI-G8


In [ ]:
# Reconstrucción de results/08_ajustes_cv.joblib con las funciones de src/pipeline.py
# (el código que lo generó no está en el repositorio). Si el archivo ya existe, no hace nada.
import joblib
from src import config as C
from src.data import dev_ids, load_split
from src.pipeline import choose_fraction, choose_value, fit_weights, training_pixels
from src.split import person_folds

archivo_ajustes = ROOT / "results" / "08_ajustes_cv.joblib"
if not archivo_ajustes.exists():
    ids = dev_ids()
    dev = load_split().set_index("image").loc[ids]
    folds = dev["person"].map(person_folds(dev.reset_index(), k=5, seed=C.SEED))
    samples = {}
    for i in ids:
        samples[i] = training_pixels(i)          # calcula los 26 mapas de cada imagen: tarda
        print(i, end=" ", flush=True)
    fits = {}
    for k in range(5):
        train = [i for i in ids if folds[i] != k]
        w = fit_weights(samples, train)
        fits[k] = {"w": w, "combinación": (choose_fraction(samples, w, train),
                                           choose_value(samples, w, train))}
        print(f"\nvuelta {k}: fracción {fits[k]['combinación'][0]}, valor {fits[k]['combinación'][1]:.4f}")
    archivo_ajustes.parent.mkdir(exist_ok=True)
    joblib.dump({"features": list(C.SEGMENT_FEATURES), "balanced": C.SEGMENT_BALANCED_TRAINING,
                 "factor": C.SEGMENT_FACTOR, "fits": fits, "folds": folds}, archivo_ajustes)
    print("Guardado:", archivo_ajustes)

# Etapa 8 — Robustez

Se evalúa cómo cambia el desempeño del algoritmo al agregar
compresión JPEG, desenfoque y ruido a las imágenes de desarrollo.

Se comparan las reglas de fracción fija y valor fijo, manteniendo
los modelos sin reajustarlos a las degradaciones. También se
evalúa si los pasos de restauración mejoran el resultado.

## Reglas (fijadas antes de correr el desenfoque, el ruido y la restauración)

**Declaración.** La recompresión JPEG sin restauración (celda siguiente) se corrió antes de escribir estas reglas. Sus niveles se mantienen y se evalúa con las mismas reglas que el resto.

**Algoritmo.** El final (`src/pipeline.py`), con los pesos y umbrales de la vuelta de validación cruzada en la que cada imagen quedó afuera (`08_ajustes_cv.joblib`, del 05d). No se reajusta nada a las fotos degradadas.

**Degradaciones** (`config.ROBUST_*`, `src/degrade.py`), sobre las 30 imágenes de desarrollo:
- JPEG con calidad 90, 70, 50 y 30;
- desenfoque gaussiano de σ = 1, 2 y 4 píxeles a resolución original (el límite de los vasos finos es 3,47);
- ruido gaussiano de desvío 5, 10 y 20 niveles de gris en cada canal, con una semilla fija por imagen.

**Restauración** (`restore_photo`, con las funciones de la etapa 3), aplicada a la foto degradada antes del algoritmo, una por degradación: JPEG → suavizado de bloques; ruido → Wiener local; desenfoque → deconvolución de Wiener con el desenfoque conocido (balance 0,1). Que el desenfoque sea conocido favorece a la restauración: es el mejor caso posible para ella.

**Medidas.** Matthews (principal) y Dice por imagen, dentro del campo de visión de la base; con las dos reglas de umbral (fracción fija y valor fijo).

**Reglas de decisión.** Diferencias apareadas por imagen, con 2.000 remuestreos de personas y semilla 2026:
- **A. Rango en el que funciona:** un nivel funciona si el límite inferior del intervalo al 95 % de (nivel − base) es ≥ −0,03 (`NONINF_MARGIN_MATTHEWS`). El rango llega hasta el último nivel que funciona.
- **B. ¿Ayuda la restauración?** En cada nivel, (con − sin): ayuda si el límite inferior es > 0; empeora si el superior es < 0; si no, no se demuestra diferencia.
- **C. Regla del umbral** (la etapa 5b la dejó para esta etapa): se adopta el valor fijo solo si su Matthews, promediado en todos los niveles sin restauración, supera al de la fracción fija con un intervalo que no incluye el cero. Si no, se queda la fracción fija, la adoptada en la etapa 5b.
- **D.** La restauración se agrega al algoritmo final solo si ayuda (regla B) también en la base, sin degradación; si solo ayuda en fotos degradadas, se informa como recomendación para fotos de peor calidad.

In [3]:
import io
import joblib
import numpy as np
import pandas as pd
from PIL import Image
import time
import matplotlib.pyplot as plt

from src import config as C
from src.data import dev_ids, load_image, load_vessels, load_fov
from src.pipeline import feature_scores, load_dev_mask
from src.enhance import upscale
from src.fov import detect_fov_robust
from src.segment import segmentar_fraccion
from src.metrics import matthews, dice
from src.config import (FOV_DIAMETER_REF, NONINF_MARGIN_MATTHEWS, ROBUST_BLUR_SIGMA_REL,
                        ROBUST_JPEG_QUALITIES, ROBUST_NOISE_SIGMAS, SEED)
from src.data import load_split
from src.degrade import gaussian_blur, gaussian_noise, jpeg_recompress, restore_photo
from src.stats import percentile_interval, person_bootstrap_means

ajustes = joblib.load(
    ROOT / "results" / "08_ajustes_cv.joblib"
)

assert ajustes["features"] == list(C.SEGMENT_FEATURES)
assert ajustes["balanced"] == C.SEGMENT_BALANCED_TRAINING
assert ajustes["factor"] == C.SEGMENT_FACTOR

fits = ajustes["fits"]
folds = ajustes["folds"]
ids = dev_ids()

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\leahw\\PSIB\\TP-PSI-G8\\results\\08_ajustes_cv.joblib'

## Robustez frente a compresión JPEG

Se compara la fotografía base —que ya tiene compresión JPEG—
con versiones recomprimidas a calidades 90, 70, 50 y 30.

Cada imagen se evalúa con los pesos y umbrales ajustados sin
su persona en el 05d. Se mantienen ambos umbrales sin reajustarlos
a las imágenes degradadas.

En este primer experimento no se aplica restauración adicional.

In [ ]:
def recomprimir_jpeg(foto, calidad):
    """Recomprime en memoria, conservando las dimensiones."""
    archivo = io.BytesIO()
    Image.fromarray(foto).save(
        archivo, format="JPEG", quality=calidad, subsampling=2
    )
    archivo.seek(0)
    with Image.open(archivo) as imagen:
        return np.asarray(imagen.convert("RGB")).copy()


# Archivo propio de estos ajustes, para no mezclar corridas.
import hashlib

identificador = hashlib.sha256(
    (ROOT / "results" / "08_ajustes_cv.joblib").read_bytes()
).hexdigest()[:10]

salida = ROOT / "results" / f"08_jpeg_cv_{identificador}.csv"
calidades = [None, 90, 70, 50, 30]

previo = pd.read_csv(salida) if salida.exists() else pd.DataFrame()
filas = previo.to_dict("records")

for numero, imagen_id in enumerate(ids, start=1):
    foto = load_image(imagen_id)
    referencia = load_vessels(imagen_id)
    campo_ref = load_fov(imagen_id)
    ajuste = fits[int(folds.loc[imagen_id])]
    fraccion, valor = ajuste["combinación"]

    for calidad in calidades:
        nivel = "Base" if calidad is None else f"JPEG {calidad}"

        terminadas = {
            fila["Regla"] for fila in filas
            if fila["Imagen"] == imagen_id and fila["Nivel"] == nivel
        }
        if terminadas == {"Fracción fija", "Valor fijo"}:
            continue

        # La base (calidad None) pasa por el mismo camino que los niveles
        # degradados, así las diferencias se deben solo a la degradación.
        degradada = foto if calidad is None else recomprimir_jpeg(foto, calidad)

        probabilidades, _, _ = feature_scores(
            degradada, [ajuste["w"]]
        )
        puntuacion = upscale(
            probabilidades[0], referencia.shape
        )
        campo_auto = detect_fov_robust(degradada)

        mascaras = {
            "Fracción fija": segmentar_fraccion(
                puntuacion, campo_auto, fraccion
            )[0],
            "Valor fijo": (puntuacion > valor) & campo_auto,
        }

        for regla, mascara in mascaras.items():
            if regla in terminadas:
                continue

            filas.append({
                "Imagen": imagen_id,
                "Nivel": nivel,
                "Regla": regla,
                "Matthews": matthews(
                    mascara, referencia, campo_ref
                ),
                "Dice": dice(mascara, referencia, campo_ref),
            })

        pd.DataFrame(filas).to_csv(salida, index=False)
        print(
            f"{numero}/{len(ids)} — {imagen_id}: {nivel}",
            flush=True
        )

tabla_jpeg = pd.DataFrame(filas)

display(
    tabla_jpeg.groupby(["Nivel", "Regla"])[["Matthews", "Dice"]]
    .mean()
    .reindex(
        pd.MultiIndex.from_product(
            [
                ["Base", "JPEG 90", "JPEG 70", "JPEG 50", "JPEG 30"],
                ["Fracción fija", "Valor fijo"],
            ],
            names=["Nivel", "Regla"],
        )
    )
    .round(4)
)

1/30 — 01_h: Base
1/30 — 01_h: JPEG 90
1/30 — 01_h: JPEG 70
1/30 — 01_h: JPEG 50
1/30 — 01_h: JPEG 30
2/30 — 02_h: Base
2/30 — 02_h: JPEG 90
2/30 — 02_h: JPEG 70
2/30 — 02_h: JPEG 50
2/30 — 02_h: JPEG 30
3/30 — 03_h: Base
3/30 — 03_h: JPEG 90
3/30 — 03_h: JPEG 70
3/30 — 03_h: JPEG 50
3/30 — 03_h: JPEG 30
4/30 — 04_h: Base
4/30 — 04_h: JPEG 90
4/30 — 04_h: JPEG 70
4/30 — 04_h: JPEG 50
4/30 — 04_h: JPEG 30
5/30 — 07_h: Base
5/30 — 07_h: JPEG 90
5/30 — 07_h: JPEG 70
5/30 — 07_h: JPEG 50
5/30 — 07_h: JPEG 30
6/30 — 08_h: Base
6/30 — 08_h: JPEG 90
6/30 — 08_h: JPEG 70
6/30 — 08_h: JPEG 50
6/30 — 08_h: JPEG 30
7/30 — 09_h: Base
7/30 — 09_h: JPEG 90
7/30 — 09_h: JPEG 70
7/30 — 09_h: JPEG 50
7/30 — 09_h: JPEG 30
8/30 — 10_h: Base
8/30 — 10_h: JPEG 90
8/30 — 10_h: JPEG 70
8/30 — 10_h: JPEG 50
8/30 — 10_h: JPEG 30
9/30 — 13_h: Base
9/30 — 13_h: JPEG 90
9/30 — 13_h: JPEG 70
9/30 — 13_h: JPEG 50
9/30 — 13_h: JPEG 30
10/30 — 14_h: Base
10/30 — 14_h: JPEG 90
10/30 — 14_h: JPEG 70
10/30 — 14_h: JPEG 

Matthews    Dice
Nivel   Regla                          
Base    Fracción fija    0.7255  0.7468
        Valor fijo       0.7302  0.7517
JPEG 90 Fracción fija    0.7244  0.7458
        Valor fijo       0.7289  0.7505
JPEG 70 Fracción fija    0.7178  0.7398
        Valor fijo       0.7228  0.7445
JPEG 50 Fracción fija    0.7106  0.7334
        Valor fijo       0.7155  0.7375
JPEG 30 Fracción fija    0.6942  0.7185
        Valor fijo       0.7007  0.7240

### Resultado frente a compresión JPEG

El rendimiento disminuyó al reducir la calidad JPEG. La calidad
90 produjo cambios pequeños; la mayor caída apareció en calidad 30.

El valor fijo obtuvo mayores medias de Matthews y Dice en todos
los niveles. En JPEG 30 alcanzó Matthews de 0,7007, frente a
0,6942 con fracción fija.

Estas medias todavía no determinan la elección del umbral:
los intervalos por personas se calcularán junto con las otras
degradaciones. No se aplicó restauración adicional en esta corrida.

### Control: la base guardada coincide con la recalculada

La base sale de las máscaras guardadas en el 05d; los niveles degradados se recalculan con `feature_scores`. Se comprueba en 3 imágenes que recalcular la base da lo mismo, para que las diferencias se deban solo a la degradación.

In [ ]:
def segmentar_ambas(foto, ajuste, forma):
    """Máscaras con las dos reglas de umbral, con los pesos de la vuelta en la que la imagen quedó afuera."""
    fraccion, valor = ajuste["combinación"]
    (prob,), _, _ = feature_scores(foto, [ajuste["w"]])
    puntuacion = upscale(prob, forma)
    campo_auto = detect_fov_robust(foto)
    return {"Fracción fija": segmentar_fraccion(puntuacion, campo_auto, fraccion)[0],
            "Valor fijo": (puntuacion > valor) & campo_auto}


def correr_familia(familia, niveles, restauracion, solo_con=False):
    """Corre una familia de degradaciones, sin y con su restauración, y guarda un CSV.

    `niveles` es una lista de (nombre del nivel, función (foto, imagen) -> foto degradada,
    parámetros de restore_photo). El nivel "Base" sin restauración es el de la
    corrida de JPEG, así que acá no se repite.
    Con `solo_con` se corre solo con restauración (para JPEG, cuyo "sin" ya
    está en la primera corrida). Se reanuda: lo ya guardado se saltea.
    """
    salida = ROOT / "results" / f"08_{familia.lower()}_robustez_cv_{identificador}.csv"
    modos = ("con",) if solo_con else ("sin", "con")
    previo = pd.read_csv(salida) if salida.exists() else pd.DataFrame()
    filas = previo.to_dict("records")
    hechas = {(f["Imagen"], f["Nivel"], f["Restauración"]) for f in filas}

    for numero, imagen_id in enumerate(ids, start=1):
        t0 = time.time()
        foto = load_image(imagen_id)
        referencia = load_vessels(imagen_id)
        campo_ref = load_fov(imagen_id)
        ajuste = fits[int(folds.loc[imagen_id])]
        for nivel, degradar, params in niveles:
            pendientes = [r for r in modos
                          if (imagen_id, nivel, r) not in hechas and not (nivel == "Base" and r == "sin")]
            if not pendientes:
                continue
            degradada = degradar(foto, imagen_id)
            for r in pendientes:
                entrada = degradada if r == "sin" else restore_photo(degradada, restauracion, **params)
                for regla, mascara in segmentar_ambas(entrada, ajuste, referencia.shape).items():
                    filas.append({"Imagen": imagen_id, "Familia": familia, "Nivel": nivel,
                                  "Restauración": r, "Regla": regla,
                                  "Matthews": matthews(mascara, referencia, campo_ref),
                                  "Dice": dice(mascara, referencia, campo_ref)})
            pd.DataFrame(filas).to_csv(salida, index=False)
        print(f"{numero}/{len(ids)} — {imagen_id}: {time.time() - t0:.0f} s", flush=True)
    return pd.DataFrame(filas)


SIGMAS_PX = [round(r * FOV_DIAMETER_REF, 3) for r in ROBUST_BLUR_SIGMA_REL]

niveles_jpeg = [("Base", lambda f, i: f, {})] + [
    (f"JPEG {q}", lambda f, i, q=q: jpeg_recompress(f, q), {}) for q in ROBUST_JPEG_QUALITIES]
niveles_desenfoque = [
    (f"σ = {s:g} px", lambda f, i, s=s: gaussian_blur(f, s), {"blur_sigma_px": s}) for s in SIGMAS_PX]
niveles_ruido = [
    (f"σ = {s} niveles", lambda f, i, s=s: gaussian_noise(f, s, seed=SEED + sum(map(ord, i)) + s), {})
    for s in ROBUST_NOISE_SIGMAS]

In [ ]:
tabla_jpeg_con = correr_familia("JPEG", niveles_jpeg, "bloques", solo_con=True)

In [ ]:
tabla_desenfoque = correr_familia("Desenfoque", niveles_desenfoque, "wiener_desenfoque")

In [ ]:
tabla_ruido = correr_familia("Ruido", niveles_ruido, "wiener_ruido")

In [ ]:
# --- Todas las corridas juntas ------------------------------------------------
jpeg_sin = tabla_jpeg.assign(Familia="JPEG", Restauración="sin")
base = jpeg_sin[jpeg_sin["Nivel"] == "Base"]
todo = pd.concat([jpeg_sin, tabla_jpeg_con,
                  base.assign(Familia="Desenfoque"), tabla_desenfoque,
                  base.assign(Familia="Ruido"), tabla_ruido], ignore_index=True)

ORDEN = {"JPEG": ["Base"] + [f"JPEG {q}" for q in ROBUST_JPEG_QUALITIES],
         "Desenfoque": ["Base"] + [f"σ = {s:g} px" for s in SIGMAS_PX],
         "Ruido": ["Base"] + [f"σ = {s} niveles" for s in ROBUST_NOISE_SIGMAS]}
personas = load_split().set_index("image").loc[ids, "person"]
margen = NONINF_MARGIN_MATTHEWS


def intervalo(dif):
    """Media e intervalo al 95 % de una diferencia por imagen, remuestreando personas."""
    lo, hi = percentile_interval(person_bootstrap_means(dif.to_frame(), personas.loc[dif.index]))
    return dif.mean(), float(np.ravel(lo)[0]), float(np.ravel(hi)[0])


def por_imagen(familia, regla, restauracion):
    t = todo[(todo["Familia"] == familia) & (todo["Regla"] == regla) & (todo["Restauración"] == restauracion)]
    return t.pivot(index="Imagen", columns="Nivel", values="Matthews")


# --- Regla A: rango en el que funciona (sin restauración) --------------------
filas_a = []
for familia, niveles in ORDEN.items():
    for regla in ("Fracción fija", "Valor fijo"):
        w = por_imagen(familia, regla, "sin")
        for nivel in niveles[1:]:
            media, lo, hi = intervalo(w[nivel] - w["Base"])
            filas_a.append({"Familia": familia, "Regla": regla, "Nivel": nivel,
                            "Matthews": w[nivel].mean(), "Diferencia con la base": media,
                            "Límite inferior": lo, "Límite superior": hi, "Funciona": lo >= -margen})
rango = pd.DataFrame(filas_a)
display(rango.round(4))

# --- Regla B: ¿ayuda la restauración? -----------------------------------------
filas_b = []
for familia, niveles in ORDEN.items():
    for regla in

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (familia, niveles) in zip(axes, ORDEN.items()):
    for restauracion, estilo in (("sin", "o-"), ("con", "s--")):
        w = por_imagen(familia, REGLA, restauracion).reindex(columns=niveles)
        if restauracion == "con" and familia != "JPEG":
            w["Base"] = por_imagen(familia, REGLA, "sin")["Base"]
        ax.plot(range(len(niveles)), w.mean(), estilo, label=f"{restauracion} restauración")
    ax.set_xticks(range(len(niveles)), niveles, rotation=20)
    ax.set_title(familia)
    ax.grid(alpha=0.3)
axes[0].set_ylabel(f"Matthews medio ({REGLA.lower()})")
axes[0].legend()
fig.suptitle("Robustez: desempeño del algoritmo frente a degradaciones crecientes (30 imágenes de desarrollo)")
plt.tight_layout()
plt.show()

## Decisión (escrita después de correr)

1. **Rango en el que funciona:** JPEG hasta …; desenfoque hasta σ = …; ruido hasta σ = ….
2. **Restauración:** suavizado de bloques …; Wiener contra el ruido …; Wiener contra el desenfoque ….
3. **Regla del umbral:** … (diferencia …, intervalo …). Se actualiza `config.py`.
4. **¿Se agrega restauración al algoritmo final?** … (regla D).
5. **Pesos reconstruidos:** … (coinciden o no con los originales de la compañera).
6. **Limitaciones:** las degradaciones son simuladas sobre las mismas 30 fotos; el desenfoque se restauró conociéndolo, que es el mejor caso.